In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("books_raw.csv")
df.head()


,title,price_gbp,availability
0,A Light in the Attic,Â£51.77,In stock
1,Tipping the Velvet,Â£53.74,In stock
2,Soumission,Â£50.10,In stock
3,Sharp Objects,Â£47.82,In stock
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock


In [4]:
df["price_gbp"].dtype


dtype('O')

In [8]:
import requests

url = "https://open.er-api.com/v6/latest/GBP"
response = requests.get(url)

print(response.status_code)

200


## remving ''Â£" from price gdp columns

In [6]:
df["price_gbp"] = (
    df["price_gbp"]
    .astype(str)
    .str.replace("Â", "", regex=False)
    .str.replace("£", "", regex=False)
    .astype(float)
)


In [9]:
data = response.json()
data


{'result': 'success',
 'provider': 'https://www.exchangerate-api.com',
 'documentation': 'https://www.exchangerate-api.com/docs/free',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1770508951,
 'time_last_update_utc': 'Sun, 08 Feb 2026 00:02:31 +0000',
 'time_next_update_unix': 1770597181,
 'time_next_update_utc': 'Mon, 09 Feb 2026 00:33:01 +0000',
 'time_eol_unix': 0,
 'base_code': 'GBP',
 'rates': {'GBP': 1,
  'AED': 4.992383,
  'AFN': 88.796055,
  'ALL': 111.421842,
  'AMD': 513.516414,
  'ANG': 2.43332,
  'AOA': 1282.171658,
  'ARS': 1974.183489,
  'AUD': 1.944202,
  'AWG': 2.43332,
  'AZN': 2.309187,
  'BAM': 2.251273,
  'BBD': 2.718793,
  'BDT': 166.341594,
  'BGN': 2.206834,
  'BHD': 0.511133,
  'BIF': 4043.262136,
  'BMD': 1.359396,
  'BND': 1.729249,
  'BOB': 9.414836,
  'BRL': 7.136489,
  'BSD': 1.359396,
  'BTN': 123.043596,
  'BWP': 19.10053,
  'BYN': 3.879519,
  'BZD': 2.718793,
  'CAD': 1.857522,
  'CDF': 3107.880597,
  'CHF': 1.0556

In [12]:
gbp_to_inr = data["rates"]["INR"]
gbp_to_inr

123.044521

In [13]:
df["price_inr"] = df["price_gbp"] * gbp_to_inr



In [14]:
df["in_stock"] = df["availability"].str.contains("In stock")


In [15]:
def price_tier(price):
    if price < 500:
        return "cheap"
    elif price < 1500:
        return "moderate"
    else:
        return "expensive"

df["price_tier"] = df["price_inr"].apply(price_tier)


In [16]:
import hashlib

def generate_product_id(row):
    raw = f"{row['title']}_{row['price_gbp']}"
    return hashlib.md5(raw.encode()).hexdigest()

df["product_id"] = df.apply(generate_product_id, axis=1)


In [17]:
df.columns


Index(['title', 'price_gbp', 'availability', 'price_inr', 'in_stock',
       'price_tier', 'product_id'],
      dtype='object')

In [18]:
df.to_csv("books_raw.csv1", index=False)
